<a href="https://colab.research.google.com/github/SumitKaware/InsureAI-RAG-Based-ChatBot/blob/main/NewInsureAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install gradio google-generativeai pandas numpy tensorflow langchain langchain-community chromadb langchain-huggingface langchain-chroma langchain-google-genai

In [ ]:
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
from google.colab import userdata
import google.generativeai as genai
from google.colab import files
import os
import glob
import gradio as gr
import zipfile
import shutil
from random import randint
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

In [ ]:
api_key = userdata.get('GOOGLE_API_KEY_1')
if api_key:
    print("API Key looks good.")
else:
    print("There might be some problem with your API Key. Please check.")

MODEL = "gemini-2.0-flash-lite"
db_name = "InsureAI_VecVB"
global conversation_chain
genai.configure(api_key=api_key)
GEMINI = genai.GenerativeModel(MODEL)
prompt = "Explain how AI works in a few words"
response = GEMINI.generate_content(prompt)
response.text

In [ ]:
def process_files(uploaded_files):
    global conversation_chain
    db_name = '/content/knowledge-base'
    result_msg = ""

    if not uploaded_files:
        return "No files uploaded."

    if len(uploaded_files) == 1 and uploaded_files[0].name.endswith('.zip'):
        extract_path = '/content/uploaded_folder'
        os.makedirs(extract_path, exist_ok=True)
        with zipfile.ZipFile(uploaded_files[0].name, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        result_msg = f"[ZIP] Extracted to: {extract_path}"

        print(result_msg)
    elif len(uploaded_files) > 1:
        extract_path = '/content/uploaded_folder'
        os.makedirs(extract_path, exist_ok=True)
        for file in uploaded_files:
            shutil.move(file.name, os.path.join(extract_path, os.path.basename(file.name)))
        result_msg = f"[MULTIPLE FILES] Uploaded files moved to: {extract_path}"

        print(result_msg)
    else:
        extract_path = '/content/uploaded_folder'
        os.makedirs(extract_path, exist_ok=True)
        file = uploaded_files[0]
        shutil.move(file.name, os.path.join(extract_path, os.path.basename(file.name)))
        result_msg = f"[SINGLE FILE] Uploaded file moved to: {extract_path}"

        print(result_msg)

    processed_files = []
    for uploaded_file in uploaded_files:
        file_name = os.path.basename(uploaded_file.name)
        file_path = os.path.join(extract_path, file_name)
        shutil.copy(uploaded_file.name, file_path)
        processed_files.append(file_name)

    # Load documents using LangChain's DirectoryLoader
    text_loader_kwargs = {'autodetect_encoding': True}
    loader = DirectoryLoader(extract_path, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()

    # Assign filenames as metadata
    for doc in folder_docs:
        filename_md = os.path.basename(doc.metadata["source"])
        filename, _ = os.path.splitext(filename_md)
        doc.metadata["filename"] = filename

    documents = folder_docs
    print(documents, processed_files)
    return documents, processed_files

In [ ]:
def process_documents(uploaded_files):
    documents, processed_files = process_files(uploaded_files)
    print(documents, processed_files)
    global conversation_chain
    # Split documents into chunks
    text_splitter = CharacterTextSplitter(chunk_size=400, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)

    # Initialize embeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

    # Delete previous vectorstore
    if os.path.exists(db_name):
        Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

    print("adding files to vector db ----------\n")

    # Store in ChromaDB
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
    print("Files added ----------\n")

    # Retrieve results
    collection = vectorstore._collection
    result = collection.get(include=['embeddings', 'documents', 'metadatas'])
    return vectorstore, processed_files



In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
def chat_bot(uploaded_files):
    global conversation_chain
    vectorstore, processed_files = process_documents(uploaded_files)
    # Retrieve results
    collection = vectorstore._collection
    # result = collection.get(include=['embeddings', 'documents', 'metadatas'])
    # print("result--------------")
    # print(result)
    # Initialize LLM
    # llm = ChatGoogleGenerativeAI(temperature=0.7, model_name=MODEL)
    # llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.7)
    #llm = ChatHuggingFace(llm=MODEL)
    llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",  # Replace with desired model
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    )
    print("LLM Created ------------")
    memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
    print("Buffer memory created ------------")
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    print("Data retrived from vector db  ------------")
    conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)
    print("Data saved in conversational chain ------------")
    processed_text = "**Processed Files:**\n\n" + "\n".join(f"- {file}" for file in processed_files)
    print("Processed text from process files ------------")
    message = "Conversation Chain created"
    return message

In [ ]:
# def chat(question, history):
#     result = conversation_chain.invoke({"question": question})
#     return result["answer"]
# from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

# llm = HuggingFaceEndpoint(
#     repo_id="HuggingFaceH4/zephyr-7b-beta",
#     task="text-generation",
#     max_new_tokens=512,
#     do_sample=False,
#     repetition_penalty=1.03,
# )

# chat_model = ChatHuggingFace(llm=llm)
def chat(question, history):
    global conversation_chain  # Access the global chain
    print("inside chat window ------------")
    if conversation_chain is None:
        return "Please upload files first.", history + [(question, "Please upload files first.")]
    print("invoking chat  ------------")
    result = conversation_chain.invoke({"question": question})
    print("result--------------")
    print(result)
    response = result["answer"]
    history.append((question, response))
    return response, history

In [ ]:
def chat_with_bot(query):
    response = conversation_chain.run(query)
    return response

In [ ]:
with gr.Blocks(title="🧠 RAG Chatbot with HuggingFace LLM", theme="soft") as iface:
    chatbot = gr.Chatbot(label="AI Assistant")  # Added a Chatbot component
    textbox = gr.Textbox(placeholder="Ask me anything...")
    file_input = gr.File(file_count="multiple", label="Upload Files (ZIP or Markdown)", file_types=[".md", ".zip"])
    process_button = gr.Button("Process Files")
    history = gr.State([]) # Initialize chat history

    # Set up the main interface
    textbox.submit(
        fn=chat,
        inputs=[textbox, history],  # Corrected to 'inputs'
        outputs=[response],  # Corrected to 'outputs' and added chatbot
    )
    process_button.click(
        fn=chat_bot,
        inputs=[file_input],  # Corrected to 'inputs'
        outputs=[chatbot],  # Corrected to 'outputs' and added chatbot
    )

    iface.load(lambda: [], [], [history]) # Added history to load

iface.launch(debug=True)

In [ ]:
import inspect
def gradio_interface():

    with gr.Blocks() as demo:
        gr.Markdown("# Chat with your Documents")

        # Create a file upload component
        file_input = gr.File(
            file_count="multiple",
            label="Upload Files (ZIP or Markdown)",
            file_types=[".md", ".zip"],
        )

        # Create a chatbot component
        chatbot = gr.Chatbot(height=400)

        # Create a text input component for the query
        text_input = gr.Textbox(
            placeholder="Enter your query...",
            label="Your Query",
        )

        # Create a button to trigger the chat
        chat_button = gr.Button("Send")
        process_files_button = gr.Button("Process Files")

        # Create a state to store chat history
        output_history = gr.State([])
        output_processed_text = gr.Markdown()

        # Define what happens when the button is clicked
        process_files_button.click(
            fn=chat_bot,
            inputs=[file_input],
            outputs=[output_history],  # Store the chain
        )

        chat_button.click(
            fn=chat_with_bot,
            inputs=[text_input],
            outputs=[chatbot]
        )
        text_input.change(fn=lambda : "", inputs=[], outputs=[text_input]) #clear input box
    return demo

ui = gradio_interface()
print(f"Docstring of gradio_interface: {inspect.getdoc(gradio_interface)}")
print(f"Type of gradio_interface docstring: {type(inspect.getdoc(gradio_interface))}")
print(f"Docstring of chat_bot: {inspect.getdoc(chat_bot)}")
print(f"Type of chat_bot docstring: {type(inspect.getdoc(chat_bot))}")
print(f"Docstring of chat: {inspect.getdoc(chat)}")
print(f"Type of chat docstring: {type(inspect.getdoc(chat))}")
ui.launch(debug=True)

In [ ]:
def random_color():
        return f"rgb({randint(0,255)},{randint(0,255)},{randint(0,255)})"

In [ ]:
def show_embeddings_2d(result):
    vectors = np.array(result['embeddings'])
    documents = result['documents']
    metadatas = result['metadatas']
    filenames = [metadata['filename'] for metadata in metadatas]
    filenames_unique = sorted(set(filenames))

    # color assignment
    color_map = {name: random_color() for name in filenames_unique}
    colors = [color_map[name] for name in filenames]

    tsne = TSNE(n_components=2, random_state=42,perplexity=4)
    reduced_vectors = tsne.fit_transform(vectors)

    # Create the 2D scatter plot
    fig = go.Figure(data=[go.Scatter(
        x=reduced_vectors[:, 0],
        y=reduced_vectors[:, 1],
        mode='markers',
        marker=dict(size=5,color=colors, opacity=0.8),
        text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(filenames, documents)],
        hoverinfo='text'
    )])

    fig.update_layout(
        title='2D Chroma Vector Store Visualization',
        scene=dict(xaxis_title='x',yaxis_title='y'),
        width=800,
        height=600,
        margin=dict(r=20, b=10, l=10, t=40)
    )

    return fig


In [ ]:
def show_embeddings_3d(result):
    vectors = np.array(result['embeddings'])
    documents = result['documents']
    metadatas = result['metadatas']
    filenames = [metadata['filename'] for metadata in metadatas]
    filenames_unique = sorted(set(filenames))

    # color assignment
    color_map = {name: random_color() for name in filenames_unique}
    colors = [color_map[name] for name in filenames]

    tsne = TSNE(n_components=3, random_state=42)
    reduced_vectors = tsne.fit_transform(vectors)

    fig = go.Figure(data=[go.Scatter3d(
        x=reduced_vectors[:, 0],
        y=reduced_vectors[:, 1],
        z=reduced_vectors[:, 2],
        mode='markers',
        marker=dict(size=5, color=colors, opacity=0.8),
        text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(filenames, documents)],
        hoverinfo='text'
    )])

    fig.update_layout(
        title='3D Chroma Vector Store Visualization',
        scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
        width=900,
        height=700,
        margin=dict(r=20, b=10, l=10, t=40)
    )

    return fig

In [ ]:
def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

def visualise_data(result):
    fig_2d = show_embeddings_2d(result)
    fig_3d = show_embeddings_3d(result)
    return fig_2d,fig_3d

In [ ]:
css = """
.btn {background-color: #1d53d1;}
"""

In [ ]:
with gr.Blocks(css=css) as ui:
    gr.Markdown("# Markdown-Based Q&A with Visualization")
    with gr.Row():
        file_input=gr.File(file_types=[".zip", ".txt", ".csv", ".pdf", ".docx", ".png", ".jpg", "*"], file_count="multiple", label="Upload File(s) or ZIP Folder")
        with gr.Column(scale=1):
            processed_output = gr.Markdown("Progress")
    with gr.Row():
        process_btn = gr.Button("Process Files",elem_classes=["btn"])
    with gr.Row():
        question = gr.Textbox(label="Chat ", lines=10)
        answer = gr.Markdown(label= "Response")
    with gr.Row():
        question_btn = gr.Button("Ask a Question",elem_classes=["btn"])
        clear_btn = gr.Button("Clear Output",elem_classes=["btn"])
    with gr.Row():
        plot_2d = gr.Plot(label="2D Visualization")
        plot_3d = gr.Plot(label="3D Visualization")
    with gr.Row():
        visualise_btn = gr.Button("Visualise Data",elem_classes=["btn"])

    result = gr.State([])
    # Action: When button is clicked, process files and update visualization
    clear_btn.click(fn=lambda:("", ""), inputs=[],outputs=[question, answer])
    process_btn.click(chat_bot, inputs=[file_input], outputs=[result,processed_output])
    question_btn.click(chat, inputs=[question], outputs= [answer])
    visualise_btn.click(visualise_data, inputs=[result], outputs=[plot_2d,plot_3d])

# Launch Gradio app
ui.launch(inbrowser=True, debug=True)

In [ ]:
import gradio as gr
import os
import shutil
import zipfile
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_community.vectorstores import Chroma
# from langchain_text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter

MODEL = "models/gemini-pro"  # Or "models/gemini-pro-vision", or "models/gemini-ultra" - Define your model here
global conversation_chain

def process_documents(uploaded_files):
    """
    Processes uploaded files, extracts them if necessary, loads them as documents,
    creates a vectorstore.

    Args:
        uploaded_files (list): A list of uploaded file objects from Gradio.

    Returns:
        tuple: A tuple containing:
            - Chroma: The Chroma vectorstore.
            - list: A list of processed file names.
    """
    extract_path = '/content/uploaded_folder'
    os.makedirs(extract_path, exist_ok=True)
    processed_files = []

    if not uploaded_files:
        return None, []  # Return None and empty list for no files

    for file in uploaded_files:
        if file.name.endswith('.zip'):
            with zipfile.ZipFile(file.name, 'r') as zip_ref:
                zip_ref.extractall(extract_path)
        else:
            shutil.copy(file.name, os.path.join(extract_path, os.path.basename(file.name)))
        processed_files.append(os.path.basename(file.name))

    # Load documents using LangChain's DirectoryLoader
    text_loader_kwargs = {'autodetect_encoding': True}
    loader = DirectoryLoader(extract_path, glob="**/*.md", loader_cls=TextLoader,
                             loader_kwargs=text_loader_kwargs)
    documents = loader.load()

    # Assign filenames as metadata
    for doc in documents:
        filename_md = os.path.basename(doc.metadata["source"])
        filename, _ = os.path.splitext(filename_md)
        doc.metadata["filename"] = filename

    # Split documents
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = text_splitter.split_documents(documents)

    # Create vectorstore
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")  # Corrected
    vectorstore = Chroma.from_documents(chunks, embeddings,
                                        persist_directory='/content/knowledge-base')
    return vectorstore, processed_files



def chat_bot(uploaded_files, query, chat_history):
    """
    Processes uploaded files, retrieves results, and generates a response using a
    conversational chain.

    Args:
        uploaded_files (list): A list of uploaded file objects from Gradio.
        query (str): The user's query.
        chat_history (list): The chat history.

    Returns:
        tuple: A tuple containing:
            - str: The LLM generated response.
            - list: Updated chat history
    """
    vectorstore, processed_files = process_documents(uploaded_files)

    if vectorstore is None:
        return "Please upload files to process first.", chat_history  # Handle no files

    # Initialize LLM and chain
    llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.7)  # Corrected
    memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 35})
    conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm,
                                                              retriever=retriever,
                                                              memory=memory)

    # Generate response
    response = conversation_chain.run(query)
    chat_history.append((query, response))

    processed_text = "**Processed Files:**\n\n" + "\n".join(
        f"- {file}" for file in processed_files)

    return response, chat_history  # Return response and updated history



def gradio_interface():
    """
    Creates a Gradio interface for the chat_bot function.
    """
    with gr.Blocks() as demo:
        gr.Markdown("# Chat with your Documents")

        # Create a file upload component
        file_input = gr.File(
            file_count="multiple",
            label="Upload Files (ZIP or Markdown)",
            file_types=[".md", ".zip"],
        )

        # Create a chatbot component
        chatbot = gr.Chatbot(height=400)

        # Create a text input component for the query
        text_input = gr.Textbox(
            placeholder="Enter your query...",
            label="Your Query",
        )

        # Create a button to trigger the chat
        chat_button = gr.Button("Send")

        # Set the outputs
        output_response = gr.State()  # To store the LLM response
        output_history = gr.State([])  # To store the chat history

        # Define what happens when the button is clicked
        chat_button.click(
            fn=chat_bot,
            inputs=[file_input, text_input, output_history],
            outputs=[output_response, output_history],
        )

        #Chain the response to the chatbot
        output_response.change(fn=lambda x, y: (y[-1][1], y) if y else ("", []),
                              inputs=[output_response, output_history],
                              outputs=[chatbot, output_history])
        # Clear inputs after sending
        text_input.change(fn=lambda : "", inputs=[], outputs=[text_input])
    return demo



# if __name__ == "__main__":
#     ui = gradio_interface()
#     ui.launch(debug=True)


In [ ]:
ui = gradio_interface()
ui.launch(debug=True)

In [ ]:
import gradio as gr
import os
import shutil
import zipfile
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter

MODEL = "models/gemini-pro"  # Or "models/gemini-pro-vision", or "models/gemini-ultra" - Define your model here
global conversation_chain  # Make it global


def process_documents(uploaded_files):
    """
    Processes uploaded files, extracts them if necessary, loads them as documents,
    creates a vectorstore.

    Args:
        uploaded_files (list): A list of uploaded file objects from Gradio.

    Returns:
        tuple: A tuple containing:
            - Chroma: The Chroma vectorstore.
            - list: A list of processed file names.
    """
    extract_path = '/content/uploaded_folder'
    os.makedirs(extract_path, exist_ok=True)
    processed_files = []

    if not uploaded_files:
        return None, []  # Return None and empty list for no files

    for file in uploaded_files:
        if file.name.endswith('.zip'):
            with zipfile.ZipFile(file.name, 'r') as zip_ref:
                zip_ref.extractall(extract_path)
        else:
            shutil.copy(file.name, os.path.join(extract_path, os.path.basename(file.name)))
        processed_files.append(os.path.basename(file.name))

    # Load documents using LangChain's DirectoryLoader
    text_loader_kwargs = {'autodetect_encoding': True}
    loader = DirectoryLoader(extract_path, glob="**/*.md", loader_cls=TextLoader,
                             loader_kwargs=text_loader_kwargs)
    documents = loader.load()

    # Split documents
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = text_splitter.split_documents(documents)

    # Create vectorstore
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")  # Corrected
    vectorstore = Chroma.from_documents(chunks, embeddings,
                                        persist_directory='/content/knowledge-base')
    return vectorstore, processed_files


def chat_bot(uploaded_files):
    """
    Processes uploaded files, initializes the LLM and conversation chain.  This
    function is called once when files are uploaded.

    Args:
        uploaded_files (list): A list of uploaded file objects from Gradio.

    Returns:
        tuple: A tuple containing:
            - str:  Processed file information
            - ConversationChain:  The initialized chain.
    """
    global conversation_chain  # Use the global variable
    vectorstore, processed_files = process_documents(uploaded_files)

    if vectorstore is None:
        return "Please upload files to process first.", None  # Return None for chain

    # Initialize LLM and chain
    llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",  # Replace with desired model
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    )     # Corrected
    memory = ConversationBufferMemory(memory_key='chat_history',
                                      return_messages=True)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 15})
    conversation_chain = ConversationalRetrievalChain.from_llm(
        llm=llm, retriever=retriever, memory=memory)  # Store in global

    processed_text = "**Processed Files:**\n\n" + "\n".join(
        f"- {file}" for file in processed_files)
    return processed_text, conversation_chain  # Return chain



def chat(question, history):
    """
    Handles the conversation with the LLM, using the pre-initialized chain.

    Args:
        question (str): The user's query.
        history (list): The chat history.

    Returns:
        str: The LLM generated response.
    """
    global conversation_chain  # Access the global chain

    if conversation_chain is None:
        return "Please upload files first.", history + [(question, "Please upload files first.")]

    result = conversation_chain.invoke({"question": question})
    response = result["answer"]
    history.append((question, response))
    return response, history



def gradio_interface():
    """
    Creates a Gradio interface for the chat_bot and chat functions.
    """
    with gr.Blocks() as demo:
        gr.Markdown("# Chat with your Documents")

        # Create a file upload component
        file_input = gr.File(
            file_count="multiple",
            label="Upload Files (ZIP or Markdown)",
            file_types=[".md", ".zip"],
        )

        # Create a chatbot component
        chatbot = gr.Chatbot(height=400)

        # Create a text input component for the query
        text_input = gr.Textbox(
            placeholder="Enter your query...",
            label="Your Query",
        )

        # Create a button to trigger the chat
        chat_button = gr.Button("Send")
        process_files_button = gr.Button("Process Files")

        # Create a state to store chat history
        output_history = gr.State([])
        output_processed_text = gr.Markdown()

        # Define what happens when the button is clicked
        process_files_button.click(
            fn=chat_bot,
            inputs=[file_input],
            outputs=[output_processed_text, output_history],  # Store the chain
        )

        chat_button.click(
            fn=chat,
            inputs=[text_input, output_history],
            outputs=[chatbot, output_history],
        )
        text_input.change(fn=lambda : "", inputs=[], outputs=[text_input]) #clear input box
    return demo




ui = gradio_interface()
ui.launch(debug=True)


In [ ]:
from google.colab import userdata
HUGGINGFACE_API_KEY = userdata.get('HF_TOKEN')
HUGGINGFACE_API_KEY

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Configure the LLM endpoint
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",  # Replace with desired model
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
)

# Create a chat wrapper
chat = ChatHuggingFace(llm=llm, verbose=True)

# Messages for the LLM to process
# messages = [
#     ("system", "You are a helpful translator. Translate the user's sentence to French."),
#     ("human", "I love programming."),
# ]
messages = "What is AI?"
# Invoke the LLM with the message list
response = chat.invoke(messages)

# Output the response
print(response.content)


In [ ]:
import gradio as gr
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Set up vector store
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 35})

# Set up HuggingFace LLM (you can change repo_id to your model)
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",  # Replace with desired model
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
)
llm = ChatHuggingFace(llm=llm, verbose=True)

# Conversation memory
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# RAG chain
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

# Chat function
def chatbot_response(user_input):
    return conversation_chain.run(user_input)

# Gradio UI
gr.ChatInterface(
    fn=chatbot_response,
    title="🧠 RAG Chatbot with HuggingFace LLM",
    chatbot=gr.Chatbot(label="AI Assistant"),
    textbox=gr.Textbox(placeholder="Ask me anything...", lines=2),
    theme="soft",
    description="Ask questions and get answers from your vector DB + HuggingFace LLM!",
).launch()


In [ ]:
gr.ChatInterface(
    fn=chatbot_response,
    title="🧠 RAG Chatbot with HuggingFace LLM",
    chatbot=gr.Chatbot(label="AI Assistant"),
    textbox=gr.Textbox(placeholder="Ask me anything...", lines=2),
    theme="soft",
    description="Ask questions and get answers from your vector DB + HuggingFace LLM!",
).launch()

In [ ]:
import gradio as gr
import time  # For simulating processing

def chatbot_response(message, history, files=None):  # Added files parameter
    """
    Simulates a chatbot response.  Now can take file input.

    Args:
        message (str): The user's text input.
        history (list): The chat history.
        files (list): List of uploaded files (Gradio File objects or paths).

    Returns:
        tuple: Updated chat history.
    """
    # Simulate file processing if files are provided
    if files:
        for file in files:
            # In a real application, you would process the file here
            # (e.g., extract text, add to vector DB).
            print(f"Processing file: {file.name if hasattr(file, 'name') else file}")  #Access name
            time.sleep(2)  # Simulate processing time
        history.append([message, "Files processed.  You can now ask questions."])
        return history, None  # Clear the textbox after processing

    # Simulate a normal chat response
    if "hello" in message.lower():
        response = "Hello! How can I assist you today?"
    elif "how are you" in message.lower():
        response = "I'm doing well, thank you!"
    else:
        response = "I'm a simple chatbot.  I don't have much to say beyond processing your files and answering questions from the vectorDB."

    history.append([message, response])
    return history, None #Clear the textbox

def process_files_button_click(history, files):
    """
    This function is called when the "Process Files" button is clicked.

    Args:
       history(list): The chat history
       files (list): the files uploaded
    Returns:
        list: Updated chat history
    """
    if not files:
        return history + [["", "Please upload files before processing."]]
    for file in files:
        print(f"Processing file: {file.name if hasattr(file, 'name') else file}")
        time.sleep(2)  # Simulate file processing time
    return history + [["", "Files processed successfully!"]] #add to history

with gr.Blocks(title="🧠 RAG Chatbot with HuggingFace LLM", theme="soft") as iface: #Added theme to gr.Blocks
    chatbot = gr.Chatbot(label="AI Assistant")
    textbox = gr.Textbox(placeholder="Ask me anything...", lines=2)
    # Changed to FileDrop and added type="file"
    #file_input = gr.Files(label="Upload Files", type="file", file_count="multiple")
    file_input = gr.File(file_count="multiple", label="Upload Files (ZIP or Markdown)", file_types=[".md", ".zip"])
    process_button = gr.Button("Process Files") # added process button

    # Set up the main interface
    textbox.submit(chatbot_response, [textbox, chatbot, file_input], [chatbot, textbox]) #files is added
    process_button.click(process_files_button_click, [chatbot, file_input], [chatbot])
    file_input.change(process_files_button_click, [chatbot, file_input], [chatbot]) # Added a change event

    iface.load(lambda: [], [], [chatbot])

iface.launch()


In [ ]:
import gradio as gr
import os
import shutil
import zipfile
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
#global conversation_chain  # Make it global
from langchain.schema import Document
from langchain_chroma import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from google.colab import userdata
from google.colab import files
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

def process_files(uploaded_files):
    """
    Processes uploaded files, extracts them if necessary, loads them as documents,
    creates a vectorstore.

    Args:
        uploaded_files (list): A list of uploaded file objects from Gradio.

    Returns:
        tuple: A tuple containing:
            - Chroma: The Chroma vectorstore.
            - list: A list of processed file names.
    """
    global conversation_chain
    db_name = '/content/knowledge-base'
    processed_files = []

    if not uploaded_files:
        return None, []  # Return None and empty list for no files

    if len(uploaded_files) == 1 and uploaded_files[0].name.endswith('.zip'):
        extract_path = '/content/uploaded_folder'
        os.makedirs(extract_path, exist_ok=True)
        with zipfile.ZipFile(uploaded_files[0].name, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        result_msg = f"[ZIP] Extracted to: {extract_path}"

        print(result_msg)
    elif len(uploaded_files) > 1:
        extract_path = '/content/uploaded_folder'
        os.makedirs(extract_path, exist_ok=True)
        for file in uploaded_files:
            shutil.move(file.name, os.path.join(extract_path, os.path.basename(file.name)))
        result_msg = f"[MULTIPLE FILES] Uploaded files moved to: {extract_path}"

        print(result_msg)
    else:
        extract_path = '/content/uploaded_folder'
        os.makedirs(extract_path, exist_ok=True)
        file = uploaded_files[0]
        shutil.move(file.name, os.path.join(extract_path, os.path.basename(file.name)))
        result_msg = f"[SINGLE FILE] Uploaded file moved to: {extract_path}"

        print(result_msg)

    for uploaded_file in uploaded_files:
        file_name = os.path.basename(uploaded_file.name)
        file_path = os.path.join(extract_path, file_name)
        shutil.copy(uploaded_file.name, file_path)
        processed_files.append(file_name)

    # Load documents using LangChain's DirectoryLoader
    text_loader_kwargs = {'autodetect_encoding': True}
    loader = DirectoryLoader(extract_path, glob="**/*.md", loader_cls=TextLoader,
                             loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()

    # Assign filenames as metadata
    for doc in folder_docs:
        filename_md = os.path.basename(doc.metadata["source"])
        filename, _ = os.path.splitext(filename_md)
        doc.metadata["filename"] = filename

    documents = folder_docs
    print(documents, processed_files)
    return documents, processed_files

def process_documents(uploaded_files):
    documents, processed_files = process_files(uploaded_files)
    print(documents, processed_files)
    global conversation_chain
    # Split documents into chunks
    text_splitter = CharacterTextSplitter(chunk_size=400, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)

    # Create vectorstore
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")  # Corrected

    # Delete previous vectorstore
    if os.path.exists(db_name):
        Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

    print("adding files to vector db ----------\n")

    # Store in ChromaDB
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
    print("Files added ----------\n")

    # Retrieve results
    collection = vectorstore._collection
    result = collection.get(include=['embeddings', 'documents', 'metadatas'])
    return vectorstore, processed_files


def chat_bot(uploaded_files):
    """
    Processes uploaded files, initializes the LLM and conversation chain.  This
    function is called once when files are uploaded.

    Args:
        uploaded_files (list): A list of uploaded file objects from Gradio.

    Returns:
        tuple: A tuple containing:
            - str:  Processed file information
            - ConversationChain:  The initialized chain.
    """
    global conversation_chain  # Use the global variable
    vectorstore, processed_files = process_documents(uploaded_files)

    if vectorstore is None:
        return "Please upload files to process first.", None  # Return None for chain

    # Initialize LLM and chain
    llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",  # Replace with desired model
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    )
    print("LLM Created ------------")
    memory = ConversationBufferMemory(memory_key='chat_history',
                                      return_messages=True)
    print("Buffer memory created ------------")

    retriever = vectorstore.as_retriever(search_kwargs={"k": 15})
    print("Data retrived from vector db  ------------")
    conversation_chain = ConversationalRetrievalChain.from_llm(
        llm=llm, retriever=retriever, memory=memory)  # Store in global
    print("Data saved in conversational chain ------------")

    processed_text = "**Processed Files:**\n\n" + "\n".join(
        f"- {file}" for file in processed_files)
    print("Processed text from process files ------------")
    message = "Conversation Chain created"
    return [[None, processed_text]]  # Return chain

def chat(question, history):
    """
    Handles the conversation with the LLM, using the pre-initialized chain.

    Args:
        question (str): The user's query.
        history (list): The chat history.

    Returns:
        tuple: Updated chat history twice (once for chatbot display, once for state).
    """
    global conversation_chain
    print("inside chat window ------------")

    if conversation_chain is None:
        response = "Please upload files first."
        history.append([question, response])
        return history, history  # ✅ Return history twice

    print("invoking chat ------------")
    result = conversation_chain.invoke({"question": question})
    response = result["answer"]
    history.append([question, response])  # ✅ Ensure it's a list of lists/tuples
    return history, history  # ✅ Both chatbot and history state updated


def gradio_interface():
    """
    Creates a Gradio interface for the chat_bot and chat functions.
    """
    with gr.Blocks(title="🧠 RAG Chatbot with HuggingFace LLM", theme="soft") as iface:
        chatbot = gr.Chatbot(label="AI Assistant")
        textbox = gr.Textbox(placeholder="Ask me anything...")
        file_input = gr.File(file_count="multiple", label="Upload Files (ZIP or Markdown)", file_types=[".md", ".zip"])
        process_button = gr.Button("Process Files")
        history = gr.State([])

        textbox.submit(
            fn=chat,
            inputs=[textbox, history],
            outputs=[chatbot, history]
        )

        process_button.click(
            fn=chat_bot,
            inputs=[file_input],
            outputs=[chatbot]
        )

        iface.load(lambda: [], [], [history])

    return iface

ui = gradio_interface()
ui.launch(debug=True)



In [ ]:
def chat_bot(uploaded_files):
    """
    Initializes the LLM and conversation chain without vector store.

    Args:
        uploaded_files (list): A list of uploaded file objects from Gradio.

    Returns:
        list: A message to be shown in the chatbot.
    """
    global conversation_chain

    # Initialize LLM
    llm = HuggingFaceEndpoint(
        repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",  # Make sure this uses task="conversational"
        task="text-generation",
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
    )
    print("LLM Created ------------")

    # Memory for chat history
    memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
    print("Memory initialized ------------")

    # Create simple conversation chain (no retriever)
    conversation_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=None,  # ❌ No retriever
        memory=memory
    )
    print("Conversation chain initialized without retriever ------------")

    return [[None, "LLM initialized. You can now start chatting."]]


def chat(question, history):
    global conversation_chain
    print("inside chat window ------------")

    if conversation_chain is None:
        response = "Please initialize the chatbot first by uploading any file."
        history.append([question, response])
        return history, history

    print("invoking chat ------------")
    result = conversation_chain.invoke({"question": question})
    response = result["answer"]
    history.append([question, response])
    return history, history

def gradio_interface():
    with gr.Blocks(title="🧠 Simple LLM Chatbot", theme="soft") as iface:
        chatbot = gr.Chatbot(label="AI Assistant")
        textbox = gr.Textbox(placeholder="Ask me anything...")
        init_button = gr.Button("Start Chat")
        history = gr.State([])

        textbox.submit(fn=chat, inputs=[textbox, history], outputs=[chatbot, history])
        init_button.click(fn=chat_bot, inputs=[], outputs=[chatbot])

        iface.load(lambda: [], [], [history])

    return iface

ui = gradio_interface()
ui.launch(debug=True)

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
import gradio as gr

# Initialize global conversation chain
conversation_chain = None

def chat_bot(uploaded_files):
    """
    Initialize the LLM and create a conversational chain when files are uploaded.
    """
    global conversation_chain

    # Initialize LLM with the meta-llama model for conversational task
    llm = HuggingFaceEndpoint(
        repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",
        task="conversational",  # Correct task for conversational models
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
    )

    # Memory for chat history
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

    # Create conversation chain (no retriever here)
    conversation_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=None,  # No retrieval from documents, just chat
        memory=memory
    )

    return "Chatbot initialized! You can start chatting."

def chat(question, history):
    """
    Handles the conversation with the LLM and returns the response.
    """
    global conversation_chain

    if conversation_chain is None:
        return "Please initialize the chatbot first by uploading any file.", history

    # Invoke the model to get the response
    result = conversation_chain.invoke({"question": question})
    response = result["answer"]
    history.append([question, response])  # Append to history for context
    return history, history  # Return updated history

def gradio_interface():
    """
    Creates a Gradio interface for the chatbot.
    """
    with gr.Blocks(title="Chatbot with Llama-4", theme="soft") as iface:
        chatbot = gr.Chatbot(label="AI Assistant")
        textbox = gr.Textbox(placeholder="Ask me anything...")
        process_button = gr.Button("Initialize Chatbot")
        history = gr.State([])

        # Submit function for user input
        textbox.submit(fn=chat, inputs=[textbox, history], outputs=[chatbot, history])

        # Button to initialize the chatbot
        process_button.click(fn=chat_bot, inputs=[], outputs=[chatbot])

        # Load initial state
        iface.load(lambda: [], [], [history])

    return iface

# Launch the Gradio interface
ui = gradio_interface()
ui.launch(debug=True)


In [ ]:
import gradio as gr
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Configure the LLM endpoint
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",  # Replace with desired model
    task="conversational",  # Set to conversational task
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
)

# Create a chat wrapper
chat = ChatHuggingFace(llm=llm, verbose=True)

# Function to handle the chat and conversation history
def chat_with_bot(user_message, history):
    # Append user message to the history
    history.append(("human", user_message))

    # Invoke the model with the history
    response = chat.invoke(history)

    # Get the bot's response
    bot_message = response.content.strip()

    # Append bot message to the history
    history.append(("assistant", bot_message))

    return history, history  # Return updated history

def gradio_interface():
    """
    Creates a Gradio interface for the chatbot.
    """
    with gr.Blocks() as iface:
        chatbot = gr.Chatbot(label="AI Assistant")
        textbox = gr.Textbox(placeholder="Ask me anything...")
        history = gr.State([])  # Store chat history

        # Submit function for user input
        textbox.submit(fn=chat_with_bot, inputs=[textbox, history], outputs=[chatbot, history])

        # Add a reset button to clear the chat history
        gr.Button("Reset").click(lambda: [], outputs=history)

    return iface

# Launch the Gradio interface
ui = gradio_interface()
ui.launch(debug=True)


In [ ]:
import gradio as gr
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
import os

# Configure the LLM endpoint
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",  # Replace with desired model
    task="conversational",  # Set to conversational task
    max_new_tokens=512,
    do_sample=True,
    repetition_penalty=1.03,
)

# Create a chat wrapper
chat = ChatHuggingFace(llm=llm, verbose=True)

# Function to process documents and create a vector store
def process_documents():
    # Example document loader (Replace with your own documents)
    raw_documents = [
        {"content": "AI is the simulation of human intelligence by machines.", "metadata": {"source": "AI.txt"}},
        {"content": "Python is a programming language that allows rapid development.", "metadata": {"source": "Python.txt"}},
    ]
    documents = [Document(page_content=doc["content"], metadata=doc["metadata"]) for doc in raw_documents]

    # Create embeddings for documents using HuggingFace Embeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

    # Use Chroma to create a vector store (you can persist this vector store on disk if needed)
    vectorstore = Chroma.from_documents(documents, embeddings)

    return vectorstore

# Initialize the vector store
vectorstore = process_documents()

# Set up the retriever with Chroma
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # Retrieve top 3 most relevant documents

# Create conversation memory
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# Create the ConversationalRetrievalChain
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm, retriever=retriever, memory=memory
)

# Function to handle the chat and conversation history
def chat_with_bot(user_message, history):
    # Append user message to history
    history.append(("human", user_message))

    # Invoke the conversational chain with the history
    result = conversation_chain.invoke({"question": user_message})

    # Get the bot's response
    bot_message = result["answer"]

    # Append bot message to the history
    history.append(("assistant", bot_message))

    return history, history  # Return updated history

# Gradio interface
def gradio_interface():
    """
    Creates a Gradio interface for the chatbot.
    """
    with gr.Blocks() as iface:
        chatbot = gr.Chatbot(label="AI Assistant")
        textbox = gr.Textbox(placeholder="Ask me anything...")
        history = gr.State([])  # Store chat history

        # Submit function for user input
        textbox.submit(fn=chat_with_bot, inputs=[textbox, history], outputs=[chatbot, history])

        # Add a reset button to clear the chat history
        gr.Button("Reset").click(lambda: [], outputs=history)

    return iface

# Launch the Gradio interface
ui = gradio_interface()
ui.launch(debug=True)


In [ ]:
import gradio as gr
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document  # Import the Document class

# Configure the LLM endpoint
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-3-4b-it",  # Use the Google Gemma model
    task="conversational",  # Set the task to conversational
    max_new_tokens=512,
    do_sample=False,  # Allows for randomness in the output
    repetition_penalty=1.03,
)

# Create a chat wrapper
chat = ChatHuggingFace(llm=llm, verbose=True)

# Function to process documents and create a vector store
def process_documents():
    # Example documents as text (can replace with actual document loading logic)
    raw_documents = [
        {"content": "AI is the simulation of human intelligence by machines.", "metadata": {"source": "AI.txt"}},
        {"content": "Python is a programming language that allows rapid development.", "metadata": {"source": "Python.txt"}},
    ]

    # Convert raw documents into instances of the Document class
    documents = [Document(page_content=doc["content"], metadata=doc["metadata"]) for doc in raw_documents]

    # Create embeddings for documents using HuggingFace Embeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

    # Use Chroma to create a vector store (you can persist this vector store on disk if needed)
    vectorstore = Chroma.from_documents(documents, embeddings)

    return vectorstore

# Initialize the vector store
vectorstore = process_documents()

# Set up the retriever with Chroma
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # Retrieve top 3 most relevant documents

# Create conversation memory
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# Create the ConversationalRetrievalChain
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm, retriever=retriever, memory=memory
)

# Function to handle the chat and conversation history
async def chat_with_bot(user_message, history):
    # Append user message to history
    history.append(("human", user_message))

    # Invoke the conversational chain with the history (async)
    result = await conversation_chain.invoke({"question": user_message})

    # Get the bot's response
    bot_message = result["answer"]

    # Append bot message to the history
    history.append(("assistant", bot_message))

    return history, history  # Return updated history

# Gradio interface
def gradio_interface():
    """
    Creates a Gradio interface for the chatbot.
    """
    with gr.Blocks() as iface:
        chatbot = gr.Chatbot(label="AI Assistant")
        textbox = gr.Textbox(placeholder="Ask me anything...")
        history = gr.State([])  # Store chat history

        # Submit function for user input
        textbox.submit(fn=chat_with_bot, inputs=[textbox, history], outputs=[chatbot, history])

        # Add a reset button to clear the chat history
        gr.Button("Reset").click(lambda: [], outputs=history)

    return iface

# Launch the Gradio interface
ui = gradio_interface()
ui.launch(debug=True)


In [ ]:
import gradio as gr
from langchain_huggingface import HuggingFaceEndpoint
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.schema import Document

# Initialize global conversation chain
conversation_chain = None

def create_qa_chain(llm, vectorstore):
    """Create the question-answering chain."""
    prompt_template = """Answer the question using your own knowledge and the provided context.

Context:
{context}

Question: {question}

Previous conversation:
{chat_history}

Answer:"""

    prompt = ChatPromptTemplate.from_template(prompt_template)

    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(
            search_type="similarity", search_kwargs={"k": 3}
        ),
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": prompt},
        chain_type="stuff",
        verbose=True,
    )

def process_documents(uploaded_files):
    """
    Processes uploaded files and creates a vector store for question-answering.
    """
    # Example: If you're loading documents directly from markdown files or text files
    documents = [
        Document(page_content="AI is the simulation of human intelligence by machines.", metadata={"source": "AI.txt"}),
        Document(page_content="Python is a programming language that allows rapid development.", metadata={"source": "Python.txt"}),
    ]

    # Create embeddings for documents using HuggingFace Embeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

    # Create the vector store (Chroma is used for vector storage)
    vectorstore = Chroma.from_documents(documents, embeddings)

    return vectorstore

def chat_bot(uploaded_files):
    """
    Initialize the LLM and create a conversational chain when files are uploaded.
    """
    global conversation_chain

    # Process the uploaded files and create a vector store
    vectorstore = process_documents(uploaded_files)

    # Initialize the LLM with meta-llama model
    llm = HuggingFaceEndpoint(
        repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",
        task="conversational",  # Correct task for conversational models
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
    )

    # Create a question-answering chain using the vectorstore and LLM
    conversation_chain = create_qa_chain(llm, vectorstore)

    return "Chatbot initialized with QA functionality. You can start chatting."

def chat(question, history):
    """
    Handles the conversation with the LLM and returns the response.
    """
    global conversation_chain

    if conversation_chain is None:
        return "Please initialize the chatbot first by uploading any file.", history

    # Invoke the model to get the response using the QA chain
    result = conversation_chain.invoke({"question": question})
    response = result["answer"]

    # Append to history in the form of (sender, message) tuples
    history.append(("human", question))  # Append user input as ("human", question)
    history.append(("ai", response))  # Append LLM response as ("ai", response)

    return history, history  # Return updated history

def gradio_interface():
    """
    Creates a Gradio interface for the chatbot.
    """
    with gr.Blocks(title="Chatbot with Llama-4", theme="soft") as iface:
        chatbot = gr.Chatbot(label="AI Assistant")
        textbox = gr.Textbox(placeholder="Ask me anything...")
        process_button = gr.Button("Initialize Chatbot")
        history = gr.State([("system", "Chatbot initialized. Please ask a question.")])  # Initialize history with a system message

        # Submit function for user input
        textbox.submit(fn=chat, inputs=[textbox, history], outputs=[chatbot, history])

        # Button to initialize the chatbot
        process_button.click(fn=chat_bot, inputs=[], outputs=[chatbot])

        # Load initial state
        iface.load(lambda: [], [], [history])

    return iface

# Launch the Gradio interface
ui = gradio_interface()
ui.launch(debug=True)


In [ ]:
pip install PyPDF2

In [ ]:
import gradio as gr
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.schema import Document
from typing import List, Tuple, Optional
import PyPDF2
import zipfile
from transformers import pipeline

In [ ]:
def create_qa_chain(llm, vectorstore, memory):
    """
    Creates the question-answering chain.
    """
    prompt_template = """Answer the question using your own knowledge and the provided context.

Context:
{context}

Question: {question}

Previous conversation:
{chat_history}

Answer:"""

    prompt = ChatPromptTemplate.from_template(prompt_template)

    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(
            search_type="similarity", search_kwargs={"k": 3}
        ),
        memory=memory,
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": prompt},
        chain_type="stuff",
        verbose=True,
    )

In [ ]:
def process_documents(uploaded_files: List[gr.File]) -> Chroma:
    """
    Processes uploaded files and creates a vector store for question-answering.

    Args:
        uploaded_files: A list of uploaded files.

    Returns:
        A Chroma vector store.
    """
    documents = []
    for file in uploaded_files:
        try:
            # Basic file type handling.
            if file.name.endswith(".md"):
                with open(file.name, "r", encoding="utf-8") as f:
                    text = f.read()
                    documents.append(Document(page_content=text, metadata={"source": file.name}))
            elif file.name.endswith(".txt"):
                with open(file.name, "r", encoding="utf-8") as f:
                    text = f.read()
                    documents.append(Document(page_content=text, metadata={"source": file.name}))
            elif file.name.endswith(".pdf"):
                try:
                    with open(file.name, 'rb') as pdf_file:
                        read_pdf = PyPDF2.PdfReader(pdf_file)
                        for page in read_pdf.pages:
                            text = page.extract_text()
                            if text:  # Avoid empty pages
                                documents.append(Document(page_content=text, metadata={"source": file.name}))
                except Exception as e:
                    print(f"Error processing PDF {file.name}: {e}")
            elif file.name.endswith(".zip"):
                with zipfile.ZipFile(file.name, 'r') as zf:
                    for filename in zf.namelist():
                        if filename.endswith((".md", ".txt")):
                            try:
                                with zf.open(filename) as f:
                                    text = f.read().decode('utf-8')
                                    documents.append(Document(page_content=text, metadata={"source": filename}))
                            except Exception as e:
                                print(f"Error processing file {filename} from zip: {e}")
            else:
                print(f"Unsupported file type: {file.name}")
        except Exception as e:
            print(f"Error processing file {file.name}: {e}")

    if not documents:
        raise ValueError("No valid documents found in uploaded files.")

    # Create embeddings for documents using HuggingFace Embeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

    # Create the vector store (Chroma is used for vector storage)
    vectorstore = Chroma.from_documents(documents, embeddings)
    return vectorstore

In [ ]:
def chat_bot(uploaded_files: List[gr.File], history: List[Tuple[str, str]], memory: ConversationBufferMemory) -> Tuple[str, List[Tuple[str, str]]]:
    """
    Initialize the LLM and create a conversational chain when files are uploaded.
    """
    # Process the uploaded files and create a vector store
    try:
        vectorstore = process_documents(uploaded_files)
    except ValueError as e:
        return str(e), history  # Return the error message

    # Initialize the Hugging Face pipeline for conversational tasks
    llm = pipeline("conversational", model="meta-llama/Llama-4-Scout-17B-16E-Instruct")

    # Create a question-answering chain using the vectorstore and LLM
    qa_chain = create_qa_chain(llm, vectorstore, memory)

    # Return both a message and the updated history
    return "Chatbot initialized. You can start chatting!", history

In [ ]:
def chat(question: str, history: List[Tuple[str, str]], qa_chain: ConversationalRetrievalChain) -> Tuple[str, List[Tuple[str, str]]]:
    """
    Handles the conversation with the LLM and returns the response.

    Args:
        question: The user's question.
        history: The chat history.
        qa_chain: The initialized QA chain.

    Returns:
        A tuple containing the response and the updated chat history.
    """

    if qa_chain is None:
        return "Please initialize the chatbot first by uploading files and clicking 'Initialize Chatbot'.", history

    try:
        # Invoke the model to get the response using the QA chain
        result = qa_chain({"question": question, "chat_history": history})
        response = result["answer"]
        # Append to history in the form of (sender, message) tuples
        history.append(("human", question))  # Append user input
        history.append(("ai", response))  # Append LLM response
        return response, history  # Return response
    except Exception as e:
        return f"An error occurred: {e}", history


In [ ]:
def gradio_interface():
    """
    Creates a Gradio interface for the chatbot.
    """
    with gr.Blocks(title="Chatbot with Llama-4", theme="soft") as iface:
        chatbot = gr.Chatbot(label="AI Assistant")
        textbox = gr.Textbox(placeholder="Ask me anything...")
        file_input = gr.File(file_count="multiple", label="Upload Files (ZIP or Markdown)", file_types=[".md", ".zip", ".pdf", ".txt"])
        process_button = gr.Button("Initialize Chatbot")
        history = gr.State([])
        memory = gr.State(ConversationBufferMemory(memory_key='chat_history', return_messages=False))  # Initialize memory in state

        # Initialize the LLM outside the event handler for persistence
        llm = HuggingFaceEndpoint(
            repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",
            task="conversational",  # Correct task for conversational models
            model_kwargs={
                "max_new_tokens": 512,
                "do_sample": False,
                "repetition_penalty": 1.03,
            },
        )

        # Button to initialize the chatbot
        process_button.click(
            fn=chat_bot,
            inputs=[file_input, history, memory],  # Pass file input
            outputs=[chatbot, history],  # Return chatbot and history as expected
        )

        # Submit function for user input
        textbox.submit(
            fn=chat,
            inputs=[textbox, history, gr.State()],
            outputs=[chatbot, history],
        )

        # Load initial state
        iface.load(lambda: [], [], [history, memory])

    return iface


# Launch the Gradio interface
ui = gradio_interface()
ui.launch(debug=True)

In [ ]:
pip install langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 15.7 MB/s eta 0:00:00


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from typing import List
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.documents import Document

# 1. Load data and create embeddings
# (Replace with your actual data loading and embedding logic)
# def load_data():
#     """Loads your data and returns a list of LangChain documents."""
#     # Placeholder: Replace with your data loading logic
#     # Example:
#     # from langchain_document_loaders import TextLoader
#     # documents = loader.load()
#     # return documents
#     return [
#         Document(page_content="This is a sample document about LangChain.", metadata={"source": "doc1"}),
#         Document(page_content="LangGraph is a way to build LLM applications.", metadata={"source": "doc2"}),
#         Document(page_content="Chroma is a vector database.", metadata={"source": "doc3"})
#     ]

def create_vectorstore(documents: List[Document], embeddings: HuggingFaceEmbeddings) -> Chroma:
    """
    Creates a Chroma vector store from the given documents and embeddings.

    Args:
        documents: A list of LangChain documents.
        embeddings: The HuggingFaceEmbeddings to use.

    Returns:
        A Chroma vector store.
    """
    return Chroma.from_documents(documents, embeddings)



# 2.  Set up the Retriever Node
def get_relevant_documents(query, vectorstore):
    """
    Retrieves relevant documents from the vectorstore based on the query.

    Args:
        query: The user's query.
        vectorstore: The Chroma vector store.

    Returns:
        A list of relevant LangChain documents.
    """
    retriever = vectorstore.as_retriever()
    return retriever.get_relevant_documents(query)



# 3.  Set up the Generator Node
#from langchain_llm import Ollama
from langchain_core.runnables import chain

# 3. Set up the LLM for response generation
llm = HuggingFaceEndpoint(repo_id="meta-llama/Llama-4-Scout-17B-16E-Instruct")  # Replace "mistral" with your desired model
#llm = Ollama(model="mistral")  # Replace "mistral" with your desired model

# 4. Define the prompt template
prompt = PromptTemplate.from_template(
    """Answer the question based on the context:
{context}

Question: {question}
"""
)
def generate_response(query, documents):
    context = "\n\n".join(doc.page_content for doc in documents)
    return llm.invoke(prompt.format(context=context, question=query))


# 4. Define the Graph
import langgraph.graph as lg
from langchain_core.documents import Document

def rag_graph(vectorstore):
    """
    Sets up the LangGraph graph for RAG.
    """
    # 5. Create a new graph
    graph = lg.Graph()

    # 6. Add nodes
    graph.add_node("retriever", lambda x: get_relevant_documents(x, vectorstore))
    graph.add_node("generator", generate_response)

    # 7. Add edges
    graph.add_edge("retriever", "generator")
    graph.set_entry_point("retriever")

    # 8. Compile the graph
    chain = graph.compile()
    return chain

def run_rag_graph(query: str, vectorstore: Chroma):
    """
    Runs the RAG graph with the given query and vectorstore.

    Args:
        query: The user's query.
        vectorstore: The Chroma vectorstore.

    Returns:
        The generated response.
    """
    bound_rag_graph = rag_graph(vectorstore) # Bind the vectorstore
    result = bound_rag_graph.invoke(query)
    return result
if __name__ == "__main__":
    # 0.  Initialize
    embeddings = HuggingFaceEmbeddings()
    documents = [
        Document(page_content="This is a sample document about LangChain.", metadata={"source": "doc1"}),
        Document(page_content="LangGraph is a way to build LLM applications.", metadata={"source": "doc2"}),
        Document(page_content="Chroma is a vector database.", metadata={"source": "doc3"})
    ]
    vectorstore = create_vectorstore(documents, embeddings) # Create vectorstore.
    # 9. Run the graph
    query = "What is LangGraph?"
    result = run_rag_graph(query, vectorstore)
    print(f"Result: {result}")



<ipython-input-17-a6ab5138f43c>:118: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


TypeError: generate_response() missing 1 required positional argument: 'documents'